# TDGraph-IDS -- Phase 1: Data Fusion

**Goal:** Load NF-UNSW-NB15-v2 and CIC-IDS-2018, map both to a unified schema, align attack label taxonomy, clean data quality issues, and save a single merged dataset for all downstream phases.

**Inputs:**
- `data/raw/NF-UNSW-NB15-v2.csv`
- `data/raw/CIC-IDS-2018/*.csv`

**Outputs:**
- `data/processed/unified_flows.csv`
- `data/processed/attack_taxonomy.csv`
- `outputs/plots/phase1_*.png`

**Research context:**  
NF-UNSW-NB15-v2 provides NetFlow-native features (IP, port, bytes, packets) that map directly to our graph construction pipeline. CIC-IDS-2018 adds richer flow statistics (inter-arrival times, TCP flag counts) and a broader attack category vocabulary. Together they cover 12 unified attack classes.  
Prior work (E-GraphSAGE, TE-G-SAGE) uses NF-UNSW alone. Using both datasets increases attack diversity and strengthens the evaluation.

---

## 1.1 -- Setup and path validation

In [1]:
import sys
import os
import glob
import warnings
import logging

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

warnings.filterwarnings('ignore')

# Add project root to path so config.py is importable
# from any working directory
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import CFG, ensure_dirs

ensure_dirs()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('phase1')

log.info('Project root : %s', CFG.ROOT)
log.info('NF-UNSW path : %s', CFG.NF_PATH)
log.info('CIC-2018 dir : %s', CFG.CIC_DIR)
log.info('Output dir   : %s', CFG.DATA_PROCESSED)

22:51:43  INFO  Project root : C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS
22:51:43  INFO  NF-UNSW path : C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\data\raw\NF-UNSW-NB15-v2.csv
22:51:43  INFO  CIC-2018 dir : C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\data\raw\CIC-IDS-2018
22:51:43  INFO  Output dir   : C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\data\processed


## 1.2 -- Validate dataset files

Fail fast with a clear message if either dataset is missing. This prevents confusing downstream errors.

CIC-IDS-2018 absence is logged as a warning, not an error -- the notebook can run on NF-UNSW alone if needed.

In [2]:
def validate_dataset_files() -> dict:
    """
    Check that raw dataset files exist before attempting to load them.

    Returns
    -------
    dict
        Keys: 'nf_ok' (bool), 'cic_ok' (bool), 'cic_files' (list).

    Raises
    ------
    FileNotFoundError
        If NF-UNSW-NB15-v2.csv is missing.
    """
    result = {'nf_ok': False, 'cic_ok': False, 'cic_files': []}

    if CFG.NF_PATH.exists():
        size_mb = CFG.NF_PATH.stat().st_size / 1e6
        log.info('NF-UNSW-NB15-v2.csv found (%.1f MB)', size_mb)
        result['nf_ok'] = True
    else:
        raise FileNotFoundError(
            f'NF-UNSW-NB15-v2.csv not found at {CFG.NF_PATH}. '
            'Place the file in data/raw/ and re-run.'
        )

    cic_files = sorted(glob.glob(str(CFG.CIC_DIR / '*.csv')))
    if cic_files:
        log.info('CIC-IDS-2018 found: %d CSV files', len(cic_files))
        result['cic_ok'] = True
        result['cic_files'] = cic_files
    else:
        log.warning(
            'CIC-IDS-2018 not found at %s. Proceeding with NF-UNSW only.',
            CFG.CIC_DIR,
        )

    return result


file_status = validate_dataset_files()

print('NF-UNSW present :', file_status['nf_ok'])
print('CIC-2018 present:', file_status['cic_ok'])
print('CIC files found :', len(file_status['cic_files']))

22:52:41  INFO  NF-UNSW-NB15-v2.csv found (441.9 MB)
22:52:41  INFO  CIC-IDS-2018 found: 3 CSV files


NF-UNSW present : True
CIC-2018 present: True
CIC files found : 3


## 1.3 -- Unified schema definition

Both datasets use different column names for the same concepts. Explicit mapping dictionaries make the transformation auditable and easy to extend. The unified schema keeps 20+ columns present or derivable in both datasets. Columns unique to one dataset are preserved and filled with NaN in the other.

In [3]:
# NF-UNSW-NB15-v2 column -> unified column name
NF_COLUMN_MAP = {
    'IPV4_SRC_ADDR'             : 'src_ip',
    'IPV4_DST_ADDR'             : 'dst_ip',
    'L4_SRC_PORT'               : 'src_port',
    'L4_DST_PORT'               : 'dst_port',
    'PROTOCOL'                  : 'protocol',
    'L7_PROTO'                  : 'l7_proto',
    'IN_BYTES'                  : 'fwd_bytes',
    'IN_PKTS'                   : 'fwd_pkts',
    'OUT_BYTES'                 : 'bwd_bytes',
    'OUT_PKTS'                  : 'bwd_pkts',
    'FLOW_DURATION_MILLISECONDS': 'duration_ms',
    'TCP_FLAGS'                 : 'tcp_flags',
    'CLIENT_TCP_FLAGS'          : 'client_tcp_flags',
    'SERVER_TCP_FLAGS'          : 'server_tcp_flags',
    'DURATION_IN'               : 'duration_in',
    'DURATION_OUT'              : 'duration_out',
    'MIN_TTL'                   : 'min_ttl',
    'MAX_TTL'                   : 'max_ttl',
    'Label'                     : 'label',
    'Attack'                    : 'attack_raw',
}

# CIC-IDS-2018 column -> unified column name
CIC_COLUMN_MAP = {
    'Src IP'            : 'src_ip',
    'Dst IP'            : 'dst_ip',
    'Src Port'          : 'src_port',
    'Dst Port'          : 'dst_port',
    'Protocol'          : 'protocol',
    'Flow Duration'     : 'duration_ms',
    'Tot Fwd Pkts'      : 'fwd_pkts',
    'Tot Bwd Pkts'      : 'bwd_pkts',
    'TotLen Fwd Pkts'   : 'fwd_bytes',
    'TotLen Bwd Pkts'   : 'bwd_bytes',
    'Flow Byts/s'       : 'flow_bytes_per_sec',
    'Flow Pkts/s'       : 'flow_pkts_per_sec',
    'Flow IAT Mean'     : 'iat_mean',
    'Flow IAT Std'      : 'iat_std',
    'Flow IAT Max'      : 'iat_max',
    'Flow IAT Min'      : 'iat_min',
    'SYN Flag Cnt'      : 'syn_flag_cnt',
    'RST Flag Cnt'      : 'rst_flag_cnt',
    'PSH Flag Cnt'      : 'psh_flag_cnt',
    'ACK Flag Cnt'      : 'ack_flag_cnt',
    'URG Flag Cnt'      : 'urg_flag_cnt',
    'Init Fwd Win Byts' : 'init_fwd_win',
    'Init Bwd Win Byts' : 'init_bwd_win',
    'Label'             : 'attack_raw',
}

# Columns required for Phase 3 graph construction
GRAPH_REQUIRED_COLS = {
    'src_ip', 'dst_ip', 'src_port', 'dst_port',
    'fwd_bytes', 'fwd_pkts', 'bwd_bytes', 'bwd_pkts',
    'duration_ms', 'protocol', 'label', 'attack_raw',
}

log.info('NF-UNSW map  : %d columns', len(NF_COLUMN_MAP))
log.info('CIC-2018 map : %d columns', len(CIC_COLUMN_MAP))

22:53:00  INFO  NF-UNSW map  : 20 columns
22:53:00  INFO  CIC-2018 map : 24 columns


## 1.4 -- Attack label taxonomy

Both datasets use inconsistent attack category names. A single lookup table maps every raw label to one of 12 unified categories. Unmapped labels receive 'Other' and are flagged in the summary.

In [4]:
ATTACK_TAXONOMY = {
    'benign'                     : 'Benign',
    'normal'                     : 'Benign',
    'dos'                        : 'DoS',
    'dos hulk'                   : 'DoS',
    'dos goldeneye'              : 'DoS',
    'dos slowloris'              : 'DoS',
    'dos slowhttptest'           : 'DoS',
    'ddos'                       : 'DDoS',
    'ddos attack-hoic'           : 'DDoS',
    'ddos attack-loic-udp'       : 'DDoS',
    'ddos attack-loic-http'      : 'DDoS',
    'brute force'                : 'BruteForce',
    'ftp-bruteforce'             : 'BruteForce',
    'ssh-bruteforce'             : 'BruteForce',
    'brute force -web'           : 'BruteForce',
    'brute force -xss'           : 'WebAttack',
    'web attack -- brute force'  : 'WebAttack',
    'web attack -- xss'          : 'WebAttack',
    'web attack -- sql injection': 'WebAttack',
    'sql injection'              : 'WebAttack',
    'xss'                        : 'WebAttack',
    'reconnaissance'             : 'Reconnaissance',
    'portscan'                   : 'Reconnaissance',
    'infilteration'              : 'Infiltration',
    'infiltration'               : 'Infiltration',
    'exploits'                   : 'Exploits',
    'heartbleed'                 : 'Exploits',
    'bot'                        : 'Malware',
    'backdoor'                   : 'Malware',
    'shellcode'                  : 'Malware',
    'worms'                      : 'Malware',
    'fuzzers'                    : 'Fuzzers',
    'generic'                    : 'Generic',
    'analysis'                   : 'Analysis',
}


def apply_taxonomy(raw_label: str) -> str:
    """
    Map a raw attack label to a unified taxonomy category.

    Parameters
    ----------
    raw_label : str
        Original label string from either dataset.

    Returns
    -------
    str
        Unified category name, or 'Other' if not in taxonomy.
    """
    return ATTACK_TAXONOMY.get(str(raw_label).strip().lower(), 'Other')


log.info('Taxonomy defined: %d mappings -> %d categories',
         len(ATTACK_TAXONOMY), len(set(ATTACK_TAXONOMY.values())))

print('Unified categories:', sorted(set(ATTACK_TAXONOMY.values())))

22:53:51  INFO  Taxonomy defined: 34 mappings -> 12 categories


Unified categories: ['Analysis', 'Benign', 'BruteForce', 'DDoS', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Infiltration', 'Malware', 'Reconnaissance', 'WebAttack']


## 1.5 -- Load NF-UNSW-NB15-v2

In [5]:
def load_nf_unsw(path: str) -> pd.DataFrame:
    """
    Load NF-UNSW-NB15-v2 and apply unified schema mapping.

    Steps:
        1. Read CSV with low_memory=False.
        2. Keep and rename only mapped columns.
        3. Ensure binary label exists.
        4. Apply attack taxonomy.
        5. Tag source dataset.
        6. Validate graph-required columns are present.

    Parameters
    ----------
    path : str
        Absolute path to NF-UNSW-NB15-v2.csv.

    Returns
    -------
    pd.DataFrame
        Unified schema dataframe.

    Raises
    ------
    ValueError
        If mandatory graph columns are missing after mapping.
    """
    log.info('Loading NF-UNSW-NB15-v2 ...')
    raw = pd.read_csv(path, low_memory=False)
    log.info('  Raw shape: %d rows x %d columns', *raw.shape)

    available_cols = [c for c in NF_COLUMN_MAP if c in raw.columns]
    missing_cols   = [c for c in NF_COLUMN_MAP if c not in raw.columns]

    if missing_cols:
        log.warning('  Columns in map but not in file: %s', missing_cols)

    df = raw[available_cols].rename(columns=NF_COLUMN_MAP).copy()

    # Ensure binary label exists
    if 'label' not in df.columns:
        if 'attack_raw' in df.columns:
            df['label'] = (
                df['attack_raw'].str.strip().str.lower() != 'benign'
            ).astype(int)
            log.info('  Derived binary label from attack_raw')
        else:
            raise ValueError(
                'Cannot derive binary label -- neither label nor '
                'attack_raw found after NF-UNSW column mapping.'
            )

    df['label'] = (
        pd.to_numeric(df['label'], errors='coerce').fillna(0).astype(int)
    )
    df['attack_unified'] = df['attack_raw'].apply(apply_taxonomy)
    df['source'] = 'NF-UNSW'

    missing_required = GRAPH_REQUIRED_COLS - set(df.columns)
    if missing_required:
        raise ValueError(
            f'Required graph columns missing after NF-UNSW mapping: '
            f'{missing_required}'
        )

    log.info('  Mapped shape: %d rows x %d columns', *df.shape)
    log.info('  Label dist  : %s', dict(df['label'].value_counts()))
    return df


nf_df = load_nf_unsw(str(CFG.NF_PATH))

print('NF-UNSW shape :', nf_df.shape)
print('Columns       :', nf_df.columns.tolist())
print()
print('Label distribution:')
print(nf_df['label'].value_counts().to_string())
print()
print('Attack categories:')
print(nf_df['attack_unified'].value_counts().to_string())

22:54:05  INFO  Loading NF-UNSW-NB15-v2 ...
22:54:25  INFO    Raw shape: 2390275 rows x 45 columns
22:54:27  INFO    Mapped shape: 2390275 rows x 22 columns
22:54:27  INFO    Label dist  : {0: np.int64(2295222), 1: np.int64(95053)}


NF-UNSW shape : (2390275, 22)
Columns       : ['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'l7_proto', 'fwd_bytes', 'fwd_pkts', 'bwd_bytes', 'bwd_pkts', 'duration_ms', 'tcp_flags', 'client_tcp_flags', 'server_tcp_flags', 'duration_in', 'duration_out', 'min_ttl', 'max_ttl', 'label', 'attack_raw', 'attack_unified', 'source']

Label distribution:
label
0    2295222
1      95053

Attack categories:
attack_unified
Benign            2295222
Exploits            31551
Fuzzers             22310
Generic             16560
Reconnaissance      12779
DoS                  5794
Malware              3760
Analysis             2299


## 1.6 -- Load CIC-IDS-2018

CIC-IDS-2018 is distributed as one CSV per day of capture. We load all files, apply schema mapping to each, then concatenate.

In [6]:
def load_single_cic_file(path: str) -> pd.DataFrame:
    """
    Load one CIC-IDS-2018 daily CSV and apply unified schema mapping.

    CIC files have leading whitespace in column names and may contain
    'Infinity' strings or NaN in numeric columns.

    Parameters
    ----------
    path : str
        Path to one CIC daily CSV file.

    Returns
    -------
    pd.DataFrame
        Unified schema dataframe, or empty DataFrame on read failure.
    """
    try:
        raw = pd.read_csv(path, encoding='latin1', low_memory=False)
    except Exception as exc:
        log.warning('  Failed to read %s: %s', os.path.basename(path), exc)
        return pd.DataFrame()

    # Strip whitespace from column names
    raw.columns = raw.columns.str.strip()

    available_cols = [c for c in CIC_COLUMN_MAP if c in raw.columns]
    if not available_cols:
        log.warning(
            '  No mapped columns found in %s -- skipping',
            os.path.basename(path),
        )
        return pd.DataFrame()

    df = raw[available_cols].rename(columns=CIC_COLUMN_MAP).copy()

    if 'attack_raw' in df.columns:
        df['label'] = (
            df['attack_raw'].str.strip().str.upper() != 'BENIGN'
        ).astype(int)
    else:
        df['label'] = 0

    # CIC duration is in microseconds -- convert to milliseconds
    if 'duration_ms' in df.columns:
        df['duration_ms'] = (
            pd.to_numeric(df['duration_ms'], errors='coerce').fillna(0)
            / 1000.0
        )

    df['attack_unified'] = (
        df.get('attack_raw', pd.Series(['Benign'] * len(df)))
        .apply(apply_taxonomy)
    )
    df['source'] = 'CIC-2018'
    return df


def load_cic_ids_2018(cic_files: list) -> pd.DataFrame:
    """
    Load and concatenate all CIC-IDS-2018 daily CSV files.

    Parameters
    ----------
    cic_files : list of str
        Paths to CIC daily CSV files.

    Returns
    -------
    pd.DataFrame
        Concatenated unified schema dataframe.
        Returns empty DataFrame if no files are available.
    """
    if not cic_files:
        log.warning('No CIC files provided -- returning empty dataframe')
        return pd.DataFrame()

    frames = []
    for path in cic_files:
        fname = os.path.basename(path)
        log.info('  Loading %s ...', fname)
        frame = load_single_cic_file(path)
        if not frame.empty:
            log.info('    %d rows loaded', len(frame))
            frames.append(frame)
        else:
            log.warning('    No data extracted from %s', fname)

    if not frames:
        log.warning('All CIC files failed -- returning empty dataframe')
        return pd.DataFrame()

    combined = pd.concat(frames, ignore_index=True, sort=False)
    log.info('CIC-2018 combined: %d rows x %d columns', *combined.shape)
    return combined


if file_status['cic_ok']:
    cic_df = load_cic_ids_2018(file_status['cic_files'])
    print('CIC-2018 shape:', cic_df.shape)
    if not cic_df.empty:
        print(cic_df['label'].value_counts().to_string())
else:
    cic_df = pd.DataFrame()
    print('CIC-2018 not available -- proceeding with NF-UNSW only')

22:54:47  INFO    Loading Friday-02-03-2018.csv ...
22:55:04  INFO      1048575 rows loaded
22:55:04  INFO    Loading Friday-23-02-2018.csv ...
22:55:21  INFO      1048575 rows loaded
22:55:21  INFO    Loading Thursday-15-02-2018.csv ...
22:55:39  INFO      1048575 rows loaded
22:55:39  INFO  CIC-2018 combined: 3145725 rows x 24 columns


CIC-2018 shape: (3145725, 24)
label
0    2806470
1     339255


## 1.7 -- Merge into unified dataset

Concatenate NF-UNSW and CIC into a single DataFrame. Missing columns from either dataset are filled with NaN -- downstream phases handle this during feature engineering.

In [7]:
def merge_datasets(nf: pd.DataFrame, cic: pd.DataFrame) -> pd.DataFrame:
    """
    Concatenate NF-UNSW and CIC-2018 into one unified dataset.

    Columns absent in one dataset are filled with NaN.

    Parameters
    ----------
    nf  : pd.DataFrame  NF-UNSW unified dataframe.
    cic : pd.DataFrame  CIC-2018 unified dataframe (may be empty).

    Returns
    -------
    pd.DataFrame
        Merged dataset with source column indicating origin.
    """
    frames = [nf]
    if not cic.empty:
        frames.append(cic)
        log.info(
            'Merging NF-UNSW (%d rows) + CIC-2018 (%d rows)',
            len(nf), len(cic),
        )
    else:
        log.info('Using NF-UNSW only (%d rows)', len(nf))

    merged = pd.concat(frames, ignore_index=True, sort=False)
    log.info('Merged shape: %d rows x %d columns', *merged.shape)
    return merged


unified = merge_datasets(nf_df, cic_df)

print('Merged shape:', unified.shape)
print()
print('Source distribution:')
print(unified['source'].value_counts().to_string())
print()
print('Binary label distribution:')
vc = unified['label'].value_counts()
for label, count in vc.items():
    pct = count / len(unified) * 100
    print(f'  {label} : {count:>10,}  ({pct:.2f}%)')

22:56:44  INFO  Merging NF-UNSW (2390275 rows) + CIC-2018 (3145725 rows)
22:56:45  INFO  Merged shape: 5536000 rows x 35 columns


Merged shape: (5536000, 35)

Source distribution:
source
CIC-2018    3145725
NF-UNSW     2390275

Binary label distribution:
  0 :  5,101,692  (92.15%)
  1 :    434,308  (7.85%)


## 1.8 -- Data quality cleaning

Four cleaning steps in strict order:
1. Replace inf/-inf with NaN
2. Drop rows where src_ip or dst_ip is missing (unusable for graph)
3. Remove exact duplicate rows
4. Fill remaining numeric NaN with 0
5. Clip extreme values to prevent ML overflow

Order matters: inf replacement before dedup ensures inf-to-NaN rows are also deduplicated.

In [8]:
def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply data quality cleaning to the merged unified dataset.

    Steps (in strict order):
        1. Replace inf/-inf with NaN.
        2. Drop rows with missing src_ip or dst_ip.
        3. Drop exact duplicate rows.
        4. Fill numeric NaN with 0.
        5. Clip to +-1e12 to prevent overflow in downstream ML.

    Parameters
    ----------
    df : pd.DataFrame
        Merged unified dataset.

    Returns
    -------
    pd.DataFrame
        Cleaned dataset with identical column structure.
    """
    initial_rows = len(df)
    log.info('Starting data quality cleaning on %d rows ...', initial_rows)

    df = df.replace([np.inf, -np.inf], np.nan)

    before = len(df)
    df = df.dropna(subset=['src_ip', 'dst_ip'])
    log.info(
        '  Dropped %d rows missing src_ip or dst_ip', before - len(df)
    )

    before = len(df)
    df = df.drop_duplicates()
    log.info('  Dropped %d duplicate rows', before - len(df))

    numeric_cols = df.select_dtypes(include=np.number).columns
    df[numeric_cols] = df[numeric_cols].fillna(0)
    df[numeric_cols] = df[numeric_cols].clip(-1e12, 1e12)

    final_rows = len(df)
    log.info(
        'Cleaning complete: %d -> %d rows (removed %d, %.2f%%)',
        initial_rows, final_rows,
        initial_rows - final_rows,
        (initial_rows - final_rows) / initial_rows * 100,
    )
    return df


unified = clean_dataset(unified)

print('Shape after cleaning:', unified.shape)
print()
nan_counts = unified.select_dtypes(include=np.number).isnull().sum()
nan_counts = nan_counts[nan_counts > 0]
if nan_counts.empty:
    print('No NaN values remain in numeric columns')
else:
    print('Remaining NaN in numeric columns:')
    print(nan_counts.to_string())

22:57:14  INFO  Starting data quality cleaning on 5536000 rows ...
22:57:27  INFO    Dropped 3145725 rows missing src_ip or dst_ip
22:57:31  INFO    Dropped 24639 duplicate rows
22:57:35  INFO  Cleaning complete: 5536000 -> 2365636 rows (removed 3170364, 57.27%)


Shape after cleaning: (2365636, 35)

No NaN values remain in numeric columns


## 1.9 -- Dataset summary statistics

Structured summary for the paper's data description section.

In [9]:
def print_dataset_summary(df: pd.DataFrame) -> None:
    """
    Print a structured summary of the unified dataset.

    Parameters
    ----------
    df : pd.DataFrame
        Cleaned unified dataset.
    """
    print('=' * 60)
    print('PHASE 1 -- DATASET SUMMARY')
    print('=' * 60)
    print(f'Total flows         : {len(df):>12,}')
    print(f'Total columns       : {df.shape[1]:>12,}')
    print(f'Unique src IPs      : {df["src_ip"].nunique():>12,}')
    print(f'Unique dst IPs      : {df["dst_ip"].nunique():>12,}')
    total_ips = pd.concat([df['src_ip'], df['dst_ip']]).nunique()
    print(f'Total unique IPs    : {total_ips:>12,}')
    unique_pairs = df[['src_ip', 'dst_ip']].drop_duplicates().shape[0]
    print(f'Unique IP pairs     : {unique_pairs:>12,}')
    print(f'Benign flows        : {(df["label"]==0).sum():>12,}')
    print(f'Attack flows        : {(df["label"]==1).sum():>12,}')
    print(f'Attack rate         : {df["label"].mean()*100:>11.2f}%')
    print(f'Attack categories   : {df["attack_unified"].nunique():>12,}')
    print()
    print('Source breakdown:')
    for src, count in df['source'].value_counts().items():
        print(f'  {src:<15} : {count:>10,}  ({count/len(df)*100:.1f}%)')
    print()
    print('Attack category breakdown:')
    for cat, count in df['attack_unified'].value_counts().items():
        print(f'  {cat:<20} : {count:>10,}  ({count/len(df)*100:.2f}%)')
    print('=' * 60)


print_dataset_summary(unified)

PHASE 1 -- DATASET SUMMARY
Total flows         :    2,365,636
Total columns       :           35
Unique src IPs      :           40
Unique dst IPs      :           40
Total unique IPs    :           44
Unique IP pairs     :          293
Benign flows        :    2,282,659
Attack flows        :       82,977
Attack rate         :        3.51%
Attack categories   :            8

Source breakdown:
  NF-UNSW         :  2,365,636  (100.0%)

Attack category breakdown:
  Benign               :  2,282,659  (96.49%)
  Exploits             :     31,277  (1.32%)
  Fuzzers              :     22,136  (0.94%)
  Reconnaissance       :     12,625  (0.53%)
  DoS                  :      5,640  (0.24%)
  Generic              :      5,426  (0.23%)
  Malware              :      3,636  (0.15%)
  Analysis             :      2,237  (0.09%)


## 1.10 -- Visualizations for submission

Three publication-quality plots saved to `outputs/plots/` at 200 DPI:
1. Binary class distribution
2. Unified attack category breakdown
3. Per-source label distribution (only if both datasets loaded)

No `plt.show()` calls -- all figures written directly to disk.

In [10]:
plt.rcParams.update({
    'figure.dpi'       : 200,
    'font.size'        : 11,
    'axes.titlesize'   : 12,
    'axes.titleweight' : 'bold',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})


def save_figure(fig: plt.Figure, filename: str) -> None:
    """
    Save a figure to the plots directory at 200 DPI and close it.

    Parameters
    ----------
    fig      : plt.Figure  Figure to save.
    filename : str         Output filename.
    """
    path = CFG.PLOTS / filename
    fig.savefig(path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    log.info('Saved: %s', path)


# Plot 1: Binary class distribution
label_counts = unified['label'].value_counts().sort_index()
label_names  = ['Benign', 'Attack']
bar_colors   = ['#2c7bb6', '#d7191c']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    label_names, label_counts.values,
    color=bar_colors, width=0.5,
    edgecolor='white', linewidth=0.8,
)
for bar, count in zip(bars, label_counts.values):
    pct = count / len(unified) * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + len(unified) * 0.005,
        f'{count:,}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=10,
    )
ax.set_title('Binary Class Distribution -- Unified Dataset')
ax.set_ylabel('Flow count')
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f'{int(x):,}')
)
ax.set_ylim(0, label_counts.max() * 1.18)
save_figure(fig, 'phase1_binary_class_distribution.png')


# Plot 2: Attack category distribution
cat_counts = unified['attack_unified'].value_counts()
palette    = sns.color_palette('tab10', n_colors=len(cat_counts))

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    cat_counts.index, cat_counts.values,
    color=palette, edgecolor='white', linewidth=0.6,
)
for i, (cat, count) in enumerate(cat_counts.items()):
    ax.text(
        count + len(unified) * 0.002, i,
        f'{count:,}', va='center', fontsize=9,
    )
ax.set_title('Unified Attack Category Distribution')
ax.set_xlabel('Flow count')
ax.xaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f'{int(x):,}')
)
ax.set_xlim(0, cat_counts.max() * 1.2)
ax.invert_yaxis()
save_figure(fig, 'phase1_attack_category_distribution.png')


# Plot 3: Per-source label distribution (only if both datasets present)
if unified['source'].nunique() > 1:
    src_label = (
        unified.groupby(['source', 'label'])
        .size()
        .unstack(fill_value=0)
        .rename(columns={0: 'Benign', 1: 'Attack'})
    )
    fig, ax = plt.subplots(figsize=(7, 4))
    src_label.plot(
        kind='bar', ax=ax,
        color=['#2c7bb6', '#d7191c'],
        edgecolor='white', width=0.6,
    )
    ax.set_title('Label Distribution by Dataset Source')
    ax.set_xlabel('Dataset source')
    ax.set_ylabel('Flow count')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(lambda x, _: f'{int(x):,}')
    )
    ax.legend(title='Label')
    save_figure(fig, 'phase1_source_label_distribution.png')
else:
    log.info('Single source -- source comparison plot skipped')

print('All plots saved to outputs/plots/')

22:59:21  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase1_binary_class_distribution.png
22:59:21  INFO  Saved: C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\outputs\plots\phase1_attack_category_distribution.png
22:59:21  INFO  Single source -- source comparison plot skipped


All plots saved to outputs/plots/


## 1.11 -- Save outputs

Save `unified_flows.csv` for Phase 2 and `attack_taxonomy.csv` for audit trail and reproducibility.

In [11]:
def save_phase1_outputs(df: pd.DataFrame) -> None:
    """
    Save all Phase 1 outputs to disk.

    Files written:
        data/processed/unified_flows.csv    -- main dataset for Phase 2
        data/processed/attack_taxonomy.csv  -- taxonomy mapping

    Parameters
    ----------
    df : pd.DataFrame
        Cleaned unified dataset.
    """
    df.to_csv(CFG.UNIFIED, index=False)
    size_mb = CFG.UNIFIED.stat().st_size / 1e6
    log.info(
        'Saved unified_flows.csv: %.1f MB  (%d rows)', size_mb, len(df)
    )

    taxonomy_df = (
        pd.DataFrame(
            list(ATTACK_TAXONOMY.items()),
            columns=['raw_label', 'unified_label'],
        )
        .sort_values('unified_label')
        .reset_index(drop=True)
    )
    taxonomy_path = CFG.DATA_PROCESSED / 'attack_taxonomy.csv'
    taxonomy_df.to_csv(taxonomy_path, index=False)
    log.info('Saved attack_taxonomy.csv: %d mappings', len(taxonomy_df))


save_phase1_outputs(unified)

print('=' * 60)
print('PHASE 1 COMPLETE')
print('=' * 60)
print(f'Output file : {CFG.UNIFIED}')
print(f'Shape       : {unified.shape}')
print(f'Size        : {CFG.UNIFIED.stat().st_size / 1e6:.1f} MB')
print()
print('Files written:')
for fname in ['unified_flows.csv', 'attack_taxonomy.csv']:
    p = CFG.DATA_PROCESSED / fname
    if p.exists():
        print(f'  {fname:<35} {p.stat().st_size / 1e6:.2f} MB')
print()
print('Next: run Phase2_Preprocessing.ipynb')

23:00:13  INFO  Saved unified_flows.csv: 413.0 MB  (2365636 rows)
23:00:13  INFO  Saved attack_taxonomy.csv: 34 mappings


PHASE 1 COMPLETE
Output file : C:\Users\Naveen Reddie\Desktop\TDGraph-IDS-repo\TDGraph-IDS\data\processed\unified_flows.csv
Shape       : (2365636, 35)
Size        : 413.0 MB

Files written:
  unified_flows.csv                   413.05 MB
  attack_taxonomy.csv                 0.00 MB

Next: run Phase2_Preprocessing.ipynb
